# Chapter 8: Feed-Forward and Norms

[Read this chapter online](https://jackluu.io/book/section-2-attention/ch08-feedforward-and-norms/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch08-feedforward-and-norms.ipynb)

From *Teach Your Computer to Write: Build and Train an LLM from Zero* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 8: Feed-Forward and Norms

![The map with Transformer Blocks highlighted](https://jackluu.io/book/assets/diagrams/ch08-where-we-are.png)

*Figure 8.1: With attention built, we now process that context.*

As we covered in Chapter 7, attention enables a token to gain knowledge from the surrounding tokens. But gaining that knowledge isn't enough. A token must learn what to do with that information. In this chapter you will:

* Build a feed-forward network to process context.
* Use a GELU curve to help the model learn smoothly.
* Add Layer Normalization to keep the math stable.

> **Words to Know**
>
> - **Feed-Forward**: The step after attention, where each token works on its own to process the information it gathered.
> - **GELU**: A smooth curve that replaces negative numbers with near-zero values.
> - **LayerNorm**: A step that resets numbers to a safe size so training stays stable.

## Theory

### Thinking on Your Own

Attention is like listening to your team explain their ideas, while feed-forward is going back to your desk and thinking it over yourself. 

![A token goes through a wide workspace to process information](https://jackluu.io/book/assets/diagrams/ch08-feedforward-reasoning.png)

*Figure 8.2: The network expands each token to 512 numbers, then compresses it back to 128.*

The feed-forward layer is a small neural network that operates on each token individually. As seen in Figure 8.2 we take each token which is 128 numbers long and increase it to be 512 numbers. We have more room with this 4x expansion to expand the thinking out and give it to the feed-forward network, which will return us to 128 numbers at the end.

### The Smooth GELU Curve

Inside the feed-forward network, we apply a mathematical curve called GELU. 

![ReLU turns negative numbers into exactly zero, and GELU turns them into values near zero](https://jackluu.io/book/assets/diagrams/ch08-gelu-curve.png)

*Figure 8.3: GELU replaces a hard switch with a smooth dimmer.*

The neural network has to have some non-linearity (a curved line) to be able to learn complex patterns. An older choice is ReLU. This function is like an on/off switch where anything less than 0 will equal 0 and any number greater than 0 will remain as is.

GELU works more like a dimmer switch (Figure 8.3). Anything with a high negative value will be close to 0. Numbers close to 0 will have a slight decrease and any positive number will remain about the same.

When we train our network we use math (calculus) to work out how to adjust the model's weights. Having an abrupt turn can cause the math to act unstable which the GELU function helps with by having a smoother turn. This is why you see GELU used in most current models.

### Keeping Numbers Safe with LayerNorm

When you have numbers traveling through many layers of your neural network, they can become too large or too small. Similar to compound interest once numbers start multiplying each other by a lot they can reach absurdly large or small numbers which don't allow our training to work correctly.

This is where Layer Normalization (LayerNorm) comes into play. We take all the numbers that travel through each token and normalize them. After a LayerNorm we will always have an average of 0 and `std` of 1. These numbers are printed out as `mean` and `std` in the cell below.

![A vector of varying numbers is scaled to have zero mean and unit variance](https://jackluu.io/book/assets/diagrams/ch08-layernorm.png)

*Figure 8.4: LayerNorm centers messy numbers back around zero.*

It's similar to taking a marathon race and having everyone's timer start from zero after each lap (Figure 8.4). The leader doesn't change, it just makes the values smaller. We apply LayerNorm *before* each major step in modern models, and this will allow our attention and feed-forward layers to receive better inputs.

In Figure 8.1 we take what we learned from our attention layer and run it through our feed-forward network. LayerNorm keeps the math stable before sending the output to the next layer.

## Code

```python
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        C = config.n_embd
        # A small neural network applied to each token independently
        self.net = nn.Sequential(
            nn.Linear(C, 4 * C),
            nn.GELU(),
            nn.Linear(4 * C, C),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)
```

Run the script to see the feed-forward layer and LayerNorm in action.

```text
$ python src/ch07_feedforward.py
--- GELU activation ---
GELU is like ReLU (zeros out negatives) but with a smooth curve.

  Input : [-3.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]
  GELU  : [-0.004, -0.159, -0.154, 0.0, 0.346, 0.841, 1.954, 2.996]
  ReLU  : [0.0, 0.0, 0.0, 0.0, 0.5, 1.0, 2.0, 3.0]

--- Layer Normalization ---
LayerNorm re-centers and re-scales values at each position.

Before LayerNorm: mean=5.23, std=9.13
After  LayerNorm: mean=0.0000, std=1.0039
```

**What just happened:**

1. Lines 7 and 9 build a feed-forward network that expands from 128 to 512, then back to 128.
2. Line 8 applies GELU, which gently reduces negative numbers instead of snapping them to zero.
3. We passed messy numbers through LayerNorm and saw them neatly centered at zero with a standard spread of 1.

**Shape Check:**

- Input to FeedForward: `[Batch, Time, 128]`
- Output of FeedForward: `[Batch, Time, 128]`

## Try It

> **Try It**
>
> Open `src/ch07_feedforward.py` and modify the LayerNorm input to have a very large spread: `x_single = torch.randn(config.n_embd) * 1000 + 500`. Execute the script again. Note that LayerNorm still normalizes it back to mean 0 with standard deviation of 1.
>

> **In Business**
>
> Suppose you were building some kind of ML program to generate emails for you at work, written in your company's house style. If your model is poor, it could output garbage. You can think of LayerNorm as ensuring values don't explode before heading to the next neuron layer.

## Key Takeaways

* The feed-forward layer processes each token independently to build meaning.
* It expands the token's data 4x to give the model a wider workspace.
* GELU provides a smooth mathematical curve to make training stable.
* LayerNorm resets numbers to a safe size so deep networks do not break.

## Check Your Understanding

1. Why does the feed-forward network expand its input size by 4?
2. What happens to a strongly negative number when it passes through GELU?
3. What is the average value of a token's numbers immediately after LayerNorm?

## Assignment

> **Assignment 8: Dimmer or switch**
>
> **About 15 minutes. You change one number and write no new code.** Work in this chapter's Colab notebook or on your own laptop.
>
> 1. **Run it.** Run `src/ch07_feedforward.py` as it is. Copy the three lines it prints under `GELU activation`: the `Input`, `GELU` and `ReLU` lines.
> 2. **Change one number.** Find the line `sample = torch.tensor([-3.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0])`. These are the eight test numbers the script feeds to GELU and to ReLU. Change the first one, `-3.0`, to `-2.0` and run the script again. The list now holds `-2.0` and `2.0`, two numbers of the same size with opposite signs.
> 3. **Compare.** What does GELU print for `-2.0`, and what does it print for `2.0`? What does ReLU print for the same two numbers? Which of the two, GELU or ReLU, gives `-3.0` and `-2.0` exactly the same result?
> 4. **Explain it.** In two or three sentences a manager could follow, say what GELU does to a positive number and to a negative number of the same size, and why the chapter calls GELU a dimmer and ReLU a switch.
>
> **Hand in:** both sets of three lines, your answers to step 3, and your sentences.

## Further Reading

**The line that keeps training stable.** Numbers stream through a tall stack and drift: some grow bigger, some smaller, until training plateaus. Layer normalization normalizes each token's vector to unit variance before it passes to the next layer, calculating statistics using that token's info only. So it works independently of batch size. It's just one line in Chapter 8, but one of the reasons deep Transformers can train at all.

**The smooth switch inside the feed-forward layer.** You need some sort of nonlinear step in a network, or else all your layers collapse into one layer. The common choice sets everything negative to zero, which is a hard transition. GELU smoothly fades out instead of cutting hard, which provides a smoother surface for your optimizer to work on. It's also the activation used by GPT-style models like the one from Chapter 8.


Ba, J. L., Kiros, J. R., & Hinton, G. E. (2016). *Layer normalization* (arXiv:1607.06450). arXiv. https://doi.org/10.48550/arXiv.1607.06450

Hendrycks, D., & Gimpel, K. (2016). *Gaussian error linear units (GELUs)* (arXiv:1606.08415). arXiv. https://doi.org/10.48550/arXiv.1606.08415

---

### `src/ch07_feedforward.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch07_feedforward.py"   # a cell has none, and the file uses it to find the text

"""
Implement the feed-forward network and layer normalization.
This file belongs to Chapter 8.
Run: python src/ch07_feedforward.py
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import sys

from src.utils.config import GPTConfig

# Settings
config = GPTConfig()

# --- The Idea ---
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        C = config.n_embd
        # A small neural network applied to each token independently
        self.net = nn.Sequential(
            nn.Linear(C, 4 * C),
            nn.GELU(),
            nn.Linear(4 * C, C),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 8: Feed-Forward Layer and Layer Norm\n")

    ff = FeedForward()
    total_params = sum(p.numel() for p in ff.parameters())
    print(f"FeedForward parameters: {total_params:,}")
    print(f"  (C={config.n_embd} -> 4C={4*config.n_embd} -> C={config.n_embd})")

    B, T = 2, 10
    x = torch.randn(B, T, config.n_embd)
    out = ff(x)
    print(f"\nInput  shape: {x.shape}")
    print(f"Output shape: {out.shape}   (same shape as input)")

    print("\n--- GELU activation ---")
    print("GELU is like ReLU (zeros out negatives) but with a smooth curve.")
    sample = torch.tensor([-3.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0])
    gelu_out = F.gelu(sample)
    relu_out = F.relu(sample)
    print(f"\n  Input : {sample.tolist()}")
    print(f"  GELU  : {[round(v, 3) for v in gelu_out.tolist()]}")
    print(f"  ReLU  : {relu_out.tolist()}")

    print("\n--- Layer Normalization ---")
    print("LayerNorm re-centers and re-scales values at each position.")

    # LayerNorm ensures each token's vector has mean 0 and variance 1
    # initially
    ln = nn.LayerNorm(config.n_embd)
    x_single = torch.randn(config.n_embd) * 10 + 5
    x_normed = ln(x_single)

    print(f"\nBefore LayerNorm: mean={x_single.mean():.2f}, "
          f"std={x_single.std():.2f}")
    print(f"After  LayerNorm: mean={x_normed.mean():.4f}, "
          f"std={x_normed.std():.4f}")
    print("(After normalization: mean ~0, std ~1)")

    print("\nFeed-forward and LayerNorm done! Ready for Chapter 9.")